# Praleistų reikšmių užpildymas ekstrapoliacija

**Problema:** keli stulpeliai turi NA reikšmes tik laiko eilutės pradžioje arba pabaigoje (ne viduryje).

| Stulpelis | metai | Tipas |
|---|---|---|
| Apyvarta | 2004, 2025 | ekstrapoliacija atgal + į priekį |
| Policija | 2024–2025 | ekstrapoliacija į priekį |
| Skurdas | 2004–2009 | ekstrapoliacija atgal |
| Vaikai_nesimokantys | 2004–2009 | ekstrapoliacija atgal |
| Uzmokestis | 2004–2010 | ekstrapoliacija atgal |
| alkoholio_iperkamumas | 2004–2007, 2025 | ekstrapoliacija atgal + į priekį |
| alkoholio_suvartojimas | 2025 | ekstrapoliacija į priekį |
| tabako_suvartojimas | 2025 | ekstrapoliacija į priekį |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Duomenų įkėlimas

In [2]:
df = pd.read_csv('df.csv')

In [3]:
df

,Metai,Savivaldybė,Nusikaltimai,lat,lon,Gyventoju_tankis,Bedarbiu_sk,Imigrantu_sk,Medianinis_amzius,Moteru,Apyvarta,Policija,Skurdas,Pasalpa_gaunantys,gyventoju_skaicius,Vaikai_nesimokantys,Uzmokestis,alkoholio_iperkamumas,alkoholio_suvartojimas,tabako_suvartojimas
0,2004,Akmenės rajonas,470,56.2456,22.7478,34.30,2770.0,66.0,37.0,1140.0,NaN,69.0,NaN,1557.0,28951.0,NaN,NaN,NaN,10.1,1154.0
1,2004,Alytaus miestas ir rajonas,1504,54.3963,24.0457,886.95,2490.5,71.0,38.5,1074.0,NaN,355.0,NaN,1071.5,50897.0,NaN,NaN,NaN,10.1,1154.0
2,2004,Anykščių rajonas,704,55.5167,25.1000,19.10,1017.0,14.0,42.0,1135.0,NaN,78.0,NaN,1115.0,33783.0,NaN,NaN,NaN,10.1,1154.0
3,2004,Birštonas,81,54.6167,24.0167,42.30,139.0,9.0,40.0,1194.0,NaN,22.0,NaN,143.0,5240.0,NaN,NaN,NaN,10.1,1154.0
4,2004,Biržų rajonas,615,56.2000,24.7500,23.20,1592.0,24.0,39.0,1132.0,NaN,84.0,NaN,1111.0,34266.0,NaN,NaN,NaN,10.1,1154.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1293,2025,Šiaulių rajonas,434,55.8667,23.1833,22.20,2004.0,357.0,46.0,1058.0,NaN,NaN,25.7,1146.0,40120.0,115.0,2033.3,NaN,NaN,NaN
1294,2025,Šilalės rajonas,150,55.4833,22.1833,17.30,998.0,115.0,47.0,1076.0,NaN,NaN,35.3,582.0,20531.0,140.0,1879.9,NaN,NaN,NaN
1295,2025,Šilutės rajonas,478,55.3500,21.4667,22.70,2048.0,508.0,46.0,1102.0,NaN,NaN,33.2,998.0,38181.0,361.0,2022.0,NaN,NaN,NaN
1296,2025,Širvintų rajonas,143,55.0500,24.9500,16.10,800.0,130.0,47.0,1111.0,NaN,NaN,21.1,249.0,14577.0,60.0,2153.4,NaN,NaN,NaN


## Ribų konfigūracija

In [4]:
# Logikos ribos kiekvienam stulpeliui: (min, max), None = neribota
# Minimumas > 0, nes Box-Cox transformacija reikalauja griežtai teigiamų reikšmių
RIBOS = {
    'Nusikaltimai':           (1,     None),
    'Gyventoju_tankis':       (0.01,  None),
    'Bedarbiu_sk':            (1,     None),
    'Imigrantu_sk':           (1,     None),
    'Medianinis_amzius':      (20,    85),
    'Moteru':                 (1,     None),
    'Apyvarta':               (1,     None),
    'Policija':               (1,     None),
    'Skurdas':                (0.01,  None),
    'Pasalpa_gaunantys':      (1,     None),
    'gyventoju_skaicius':     (100,   None),
    'Vaikai_nesimokantys':    (1,     None),
    'Uzmokestis':             (1,     None),
    'alkoholio_iperkamumas':  (1,     None),
    'alkoholio_suvartojimas': (0.01,  25),
    'tabako_suvartojimas':    (1,     3000),
}

# Kiek ankstyvų/vėlyvų reikšmių naudoti regresijai
# (vengiame trukdžių iš tolimų metų, pvz. finansų krizė 2008)
N_TASKAI_REGRESIJAI = 8

## Ekstrapoliacijos ir ribų tikrinimo funkcijos

In [5]:
def ekstrapoliuoti(metai_zinomi, reiksmes_zinomos, metai_truksta, n_tasku=N_TASKAI_REGRESIJAI):

    if metai_truksta[-1] < metai_zinomi[0]:
        # Trūksta pradžioje - naudojame pirmus n taškus
        x_reg = metai_zinomi[:n_tasku]
        y_reg = reiksmes_zinomos[:n_tasku]
    else:
        # Trūksta pabaigoje - naudojame paskutinius n taškų
        x_reg = metai_zinomi[-n_tasku:]
        y_reg = reiksmes_zinomos[-n_tasku:]

    slope, intercept, r, p, se = stats.linregress(x_reg, y_reg)
    return slope * np.array(metai_truksta) + intercept



def taikyti_ribas(reiksmes, col, ribos_dict, log_list, sav, metai_list):

    ribos = ribos_dict.get(col, (0, None))
    rmin, rmax = ribos
    rezultatas = reiksmes.copy().astype(float)

    for i, (val, metai) in enumerate(zip(rezultatas, metai_list)):
        orig = val
        if rmin is not None and val < rmin:
            rezultatas[i] = rmin
        if rmax is not None and val > rmax:
            rezultatas[i] = rmax
        if rezultatas[i] != orig:
            log_list.append({
                'Savivaldybė': sav,
                'Metai': metai,
                'Stulpelis': col,
                'Ekstrapoliuota': round(orig, 4),
                'Po_korekcijos': round(rezultatas[i], 4),
                'Riba_min': rmin,
                'Riba_max': rmax,
            })

    return rezultatas

## Praleistų reikšmių užpildymas

In [6]:
df_uzpildytas = df.copy()
korekciju_log = []

# Stulpeliai su praleistomis reiksmemis
skip = {'Metai', 'Savivaldybė', 'lat', 'lon'}
na_stulpeliai = [
    c for c in df.columns
    if c not in skip and df[c].isnull().any() and df[c].dtype != object
]

savivaldybes = sorted(df['Savivaldybė'].unique())
uzpildyta_sk = 0

for col in na_stulpeliai:
    for sav in savivaldybes:
        mask_sav = df_uzpildytas['Savivaldybė'] == sav
        sav_df = df_uzpildytas[mask_sav].sort_values('Metai')

        metai_visi = sav_df['Metai'].values.astype(float)
        reiksmes_visi = sav_df[col].values.astype(float)

        na_mask = np.isnan(reiksmes_visi)
        if not na_mask.any():
            continue

        zinomi_mask = ~na_mask
        if zinomi_mask.sum() < 2:
            continue

        metai_zinomi   = metai_visi[zinomi_mask]
        reiksmes_zinomos = reiksmes_visi[zinomi_mask]
        metai_truksta  = metai_visi[na_mask]

        # Gali būti NA ir pradžioje ir pabaigoje – sprendžiame atskirai
        pradzia_na = metai_truksta[metai_truksta < metai_zinomi[0]]
        pabaiga_na = metai_truksta[metai_truksta > metai_zinomi[-1]]

        ekstr_reiksmes = {}

        if len(pradzia_na) > 0:
            prog = ekstrapoliuoti(metai_zinomi, reiksmes_zinomos, pradzia_na)
            prog = taikyti_ribas(prog, col, RIBOS, korekciju_log, sav, pradzia_na)
            for m, v in zip(pradzia_na, prog):
                ekstr_reiksmes[int(m)] = v

        if len(pabaiga_na) > 0:
            prog = ekstrapoliuoti(metai_zinomi, reiksmes_zinomos, pabaiga_na)
            prog = taikyti_ribas(prog, col, RIBOS, korekciju_log, sav, pabaiga_na)
            for m, v in zip(pabaiga_na, prog):
                ekstr_reiksmes[int(m)] = v

        # Įrašome atgal į df
        for metai_val, nauja_reiksme in ekstr_reiksmes.items():
            idx = df_uzpildytas[
                (df_uzpildytas['Savivaldybė'] == sav) &
                (df_uzpildytas['Metai'] == metai_val)
            ].index
            df_uzpildytas.loc[idx, col] = round(nauja_reiksme, 4)
            uzpildyta_sk += 1

print(f'Užpildyta NA reikšmių: {uzpildyta_sk}')
print(f'Ribų korekcijų: {len(korekciju_log)}')

Užpildyta NA reikšmių: 1770
Ribų korekcijų: 5


## Kur ekstrapoliacija viršijo ribas

In [7]:
df_korekcijos = pd.DataFrame(korekciju_log)

if len(df_korekcijos) > 0:
    print('Korekcijos pagal stulpelį:')
    print(df_korekcijos.groupby('Stulpelis').size().sort_values(ascending=False).to_string())
    print()
    display(df_korekcijos.head(15))
else:
    print('Ribų pažeidimų nebuvo – visos ekstrapoliuotos reikšmės logikos diapazone.')

Korekcijos pagal stulpelį:
Stulpelis
Vaikai_nesimokantys    4
Skurdas                1



,Savivaldybė,Metai,Stulpelis,Ekstrapoliuota,Po_korekcijos,Riba_min,Riba_max
0,Šakių rajonas,2004.0,Skurdas,-1.4821,0.01,0.01,None
1,Molėtų rajonas,2004.0,Vaikai_nesimokantys,-22.4167,1.00,1.00,None
2,Molėtų rajonas,2005.0,Vaikai_nesimokantys,-14.8333,1.00,1.00,None
3,Molėtų rajonas,2006.0,Vaikai_nesimokantys,-7.2500,1.00,1.00,None
4,Molėtų rajonas,2007.0,Vaikai_nesimokantys,0.3333,1.00,1.00,None


## Išsaugojimas

In [8]:
df_uzpildytas.to_csv('df_clean_1.csv', index=False, encoding='utf-8-sig')